In [ ]:
"""
SBE decimal encoding + slack-equality penalties + rolling precision (with backtracking)

Alan model:
    x1,x2,x3,x4 >= 0, in [0,1] via encoding
    b6,b7,b8,b9 binary
    minimize:
        obj(x) = 4*x1^2 + 6*x2^2 + 10*x3^2 + 6*x1*x2 - 2*x1*x3 + 2*x2*x3
    subject to:
        (e1) x1 + x2 + x3 + x4 = 1
        (e2) 8*x1 + 9*x2 + 12*x3 + 7*x4 = 10
        (e4-e7) x_i - b_i <= 0  for i=1..4
        (e8) b6 + b7 + b8 + b9 <= 3

Slack-penalty reformulation:
    For each inequality g(x) <= 0, introduce slack s >= 0 s.t. g(x) + s = 0,
    then add penalty lambda*(g+s)^2.

    x_i - b_i <= 0  ->  x_i - b_i + s_link_i = 0 , s_link_i in [0,1]
    (b6+b7+b8+b9) <= 3 -> (b6+b7+b8+b9) + s_card = 3 , s_card in [0,3]

Rolling precision:
    State (Jx, Js). Moves include refine/backtrack of Jx and/or Js.
    Accept a move if penalized energy improves by >= theta.
    Never revisit an evaluated (Jx, Js).
"""

import time
from dataclasses import dataclass
from typing import Dict, Tuple, List, Any, Optional, Set

import dimod
from neal import SimulatedAnnealingSampler

DEC_WEIGHTS = (1, 2, 3, 3)  # SBE digit weights


# SBE encoding helpers

def sbe_affine_bits(var: str, J: int, L: float, U: float) -> Tuple[float, Dict[str, float]]:
    """
    Return affine form: x = const + sum_i coeff_i * z_i
    using SBE decimal encoding with weights (1,2,3,3) + tail bit.
    Ensures x in [L,U].
    """
    if J < 1:
        raise ValueError("J must be >= 1")
    if U < L:
        raise ValueError("Require U >= L")

    const = L
    scale = (U - L)

    coeffs: Dict[str, float] = {}
    for j in range(1, J + 1):
        place = 10 ** (-j)
        for k, w in enumerate(DEC_WEIGHTS, start=1):
            b = f"z_{var}_{j}_{k}"
            coeffs[b] = coeffs.get(b, 0.0) + scale * place * w

    tail = f"z_{var}_tail_J{J}"
    coeffs[tail] = coeffs.get(tail, 0.0) + scale * (10 ** (-J))
    return const, coeffs


def decode_from_affine(sample: Dict[str, int], const: float, coeffs: Dict[str, float]) -> float:
    val = const
    for b, a in coeffs.items():
        val += a * float(sample.get(b, 0))
    return val


# QUBO builder

def add_linear(Qlin: Dict[str, float], v: str, w: float):
    Qlin[v] = Qlin.get(v, 0.0) + w


def add_quad(Qquad: Dict[Tuple[str, str], float], u: str, v: str, w: float):
    if u == v:
        # in BQM, diagonal should be handled as linear
        raise ValueError("Diagonal quadratic requested; use add_linear instead.")
    a, b = (u, v) if u < v else (v, u)
    Qquad[(a, b)] = Qquad.get((a, b), 0.0) + w


def add_square_of_affine(
    Qlin: Dict[str, float],
    Qquad: Dict[Tuple[str, str], float],
    offset_ref: List[float],
    c0: float,
    coeffs: Dict[str, float],
    weight: float = 1.0,
):
    """
    Add: weight * (c0 + sum_i a_i z_i)^2
    Using z_i^2 = z_i.
    """
    offset_ref[0] += weight * (c0 * c0)

    items = list(coeffs.items())

    # linear: weight*(a_i^2 + 2*c0*a_i)*z_i
    for bi, ai in items:
        add_linear(Qlin, bi, weight * (ai * ai + 2.0 * c0 * ai))

    # quadratic: weight*(2*a_i*a_j)*z_i*z_j
    for i in range(len(items)):
        bi, ai = items[i]
        for j in range(i + 1, len(items)):
            bj, aj = items[j]
            add_quad(Qquad, bi, bj, weight * (2.0 * ai * aj))


def add_product_of_affines(
    Qlin: Dict[str, float],
    Qquad: Dict[Tuple[str, str], float],
    offset_ref: List[float],
    c0: float,
    coeffs_c: Dict[str, float],
    d0: float,
    coeffs_d: Dict[str, float],
    weight: float = 1.0,
):
    """
    Add: weight * (c0 + sum_i a_i z_i) * (d0 + sum_j b_j z_j)
    Expands to constant + linear + quadratic (no higher order).
    """
    # constant
    offset_ref[0] += weight * (c0 * d0)

    # linear from c0*(sum b_j z_j) + d0*(sum a_i z_i)
    for zj, bj in coeffs_d.items():
        add_linear(Qlin, zj, weight * (c0 * bj))
    for zi, ai in coeffs_c.items():
        add_linear(Qlin, zi, weight * (d0 * ai))

    # quadratic from (sum a_i z_i)*(sum b_j z_j)
    # if the same bit appears in both sets, then z*z = z -> contributes to linear
    for zi, ai in coeffs_c.items():
        for zj, bj in coeffs_d.items():
            w = weight * (ai * bj)
            if zi == zj:
                add_linear(Qlin, zi, w)  # z^2 = z
            else:
                add_quad(Qquad, zi, zj, w)


def qubo_stats(bqm: dimod.BinaryQuadraticModel) -> Dict[str, int]:
    return {
        "n_vars": len(bqm.variables),
        "n_linear": len(bqm.linear),
        "n_quadratic": len(bqm.quadratic),
    }


# Solver + rolling precision

@dataclass
class SAOptions:
    num_reads: int = 300
    sweeps: int = 3000
    seed: Optional[int] = 7


def solve_large_instance_at_precision(
    Jx: int,
    Js: int,
    lam_e1: float,
    lam_e2: float,
    lam_link: float,
    lam_card: float,
    sa: SAOptions,
) -> Dict[str, Any]:
    """
    Build and solve QUBO at fixed (Jx, Js).
    """
    sampler = SimulatedAnnealingSampler()

    # ----- variables -----
    # x1..x4 in [0,1]
    cx1, ax1 = sbe_affine_bits("x1", Jx, 0.0, 1.0)
    cx2, ax2 = sbe_affine_bits("x2", Jx, 0.0, 1.0)
    cx3, ax3 = sbe_affine_bits("x3", Jx, 0.0, 1.0)
    cx4, ax4 = sbe_affine_bits("x4", Jx, 0.0, 1.0)

    # binaries
    b6, b7, b8, b9 = "b6", "b7", "b8", "b9"

    # link slacks s6..s9 in [0,1]
    cs6, as6 = sbe_affine_bits("s6", Js, 0.0, 1.0)
    cs7, as7 = sbe_affine_bits("s7", Js, 0.0, 1.0)
    cs8, as8 = sbe_affine_bits("s8", Js, 0.0, 1.0)
    cs9, as9 = sbe_affine_bits("s9", Js, 0.0, 1.0)

    # cardinality slack sc in [0,3]
    csc, asc = sbe_affine_bits("sc", Js, 0.0, 3.0)

    Qlin: Dict[str, float] = {}
    Qquad: Dict[Tuple[str, str], float] = {}
    offset = [0.0]

    # ----- objective: 4 x1^2 + 6 x2^2 + 10 x3^2 + 6 x1 x2 - 2 x1 x3 + 2 x2 x3 -----
    add_square_of_affine(Qlin, Qquad, offset, cx1, ax1, weight=4.0)
    add_square_of_affine(Qlin, Qquad, offset, cx2, ax2, weight=6.0)
    add_square_of_affine(Qlin, Qquad, offset, cx3, ax3, weight=10.0)

    add_product_of_affines(Qlin, Qquad, offset, cx1, ax1, cx2, ax2, weight=6.0)
    add_product_of_affines(Qlin, Qquad, offset, cx1, ax1, cx3, ax3, weight=-2.0)
    add_product_of_affines(Qlin, Qquad, offset, cx2, ax2, cx3, ax3, weight=2.0)

    # ----- constraints (penalties) -----

    # e1: x1 + x2 + x3 + x4 = 1
    g0 = (cx1 + cx2 + cx3 + cx4 - 1.0)
    gcoeff: Dict[str, float] = {}
    for d in (ax1, ax2, ax3, ax4):
        for z, a in d.items():
            gcoeff[z] = gcoeff.get(z, 0.0) + a
    add_square_of_affine(Qlin, Qquad, offset, g0, gcoeff, weight=lam_e1)

    # e2: 8x1 + 9x2 + 12x3 + 7x4 = 10
    h0 = (8.0 * cx1 + 9.0 * cx2 + 12.0 * cx3 + 7.0 * cx4 - 10.0)
    hcoeff: Dict[str, float] = {}
    for z, a in ax1.items():
        hcoeff[z] = hcoeff.get(z, 0.0) + 8.0 * a
    for z, a in ax2.items():
        hcoeff[z] = hcoeff.get(z, 0.0) + 9.0 * a
    for z, a in ax3.items():
        hcoeff[z] = hcoeff.get(z, 0.0) + 12.0 * a
    for z, a in ax4.items():
        hcoeff[z] = hcoeff.get(z, 0.0) + 7.0 * a
    add_square_of_affine(Qlin, Qquad, offset, h0, hcoeff, weight=lam_e2)

    # link constraints: x_i - b_i <= 0  -> x_i - b_i + s_i = 0
    # i=1..4 corresponds to b6..b9 and s6..s9
    def add_link_penalty(cx, ax, bname: str, cs, as_, lam: float):
        r0 = (cx + cs)  # constant parts
        rcoeff: Dict[str, float] = {}
        for z, a in ax.items():
            rcoeff[z] = rcoeff.get(z, 0.0) + a
        for z, a in as_.items():
            rcoeff[z] = rcoeff.get(z, 0.0) + a
        rcoeff[bname] = rcoeff.get(bname, 0.0) - 1.0  # -b_i
        # target is 0, so just square (r0 + sum coeff*z)^2
        add_square_of_affine(Qlin, Qquad, offset, r0, rcoeff, weight=lam)

    add_link_penalty(cx1, ax1, b6, cs6, as6, lam_link)
    add_link_penalty(cx2, ax2, b7, cs7, as7, lam_link)
    add_link_penalty(cx3, ax3, b8, cs8, as8, lam_link)
    add_link_penalty(cx4, ax4, b9, cs9, as9, lam_link)

    # cardinality: b6+b7+b8+b9 <= 3 -> b6+b7+b8+b9 + sc = 3
    k0 = (csc - 3.0)
    kcoeff: Dict[str, float] = {}
    for z, a in asc.items():
        kcoeff[z] = kcoeff.get(z, 0.0) + a
    for b in (b6, b7, b8, b9):
        kcoeff[b] = kcoeff.get(b, 0.0) + 1.0
    add_square_of_affine(Qlin, Qquad, offset, k0, kcoeff, weight=lam_card)

    # ----- solve -----
    bqm = dimod.BinaryQuadraticModel(Qlin, Qquad, offset[0], vartype=dimod.BINARY)

    t0 = time.perf_counter()
    ss = sampler.sample(bqm, num_reads=sa.num_reads, sweeps=sa.sweeps, seed=sa.seed)
    t1 = time.perf_counter()

    best = ss.first.sample
    penE = float(ss.first.energy)

    # decode
    x1 = decode_from_affine(best, cx1, ax1)
    x2 = decode_from_affine(best, cx2, ax2)
    x3 = decode_from_affine(best, cx3, ax3)
    x4 = decode_from_affine(best, cx4, ax4)

    s6 = decode_from_affine(best, cs6, as6)
    s7 = decode_from_affine(best, cs7, as7)
    s8 = decode_from_affine(best, cs8, as8)
    s9 = decode_from_affine(best, cs9, as9)
    sc = decode_from_affine(best, csc, asc)

    b6v = int(best.get(b6, 0))
    b7v = int(best.get(b7, 0))
    b8v = int(best.get(b8, 0))
    b9v = int(best.get(b9, 0))

    # original objective
    obj = (
        4.0 * x1 * x1
        + 6.0 * x2 * x2
        + 10.0 * x3 * x3
        + 6.0 * x1 * x2
        - 2.0 * x1 * x3
        + 2.0 * x2 * x3
    )

    # constraint residuals/violations 
    e1_resid = (x1 + x2 + x3 + x4 - 1.0)
    e2_resid = (8.0 * x1 + 9.0 * x2 + 12.0 * x3 + 7.0 * x4 - 10.0)

    # original inequalities: x_i - b_i <= 0
    link_viol = max(0.0, x1 - b6v, x2 - b7v, x3 - b8v, x4 - b9v)

    # equality forms we penalize: x_i - b_i + s_i = 0
    link_resids = [
        (x1 - b6v + s6),
        (x2 - b7v + s7),
        (x3 - b8v + s8),
        (x4 - b9v + s9),
    ]
    link_resid_maxabs = max(abs(r) for r in link_resids)

    # cardinality: bsum <= 3
    bsum = b6v + b7v + b8v + b9v
    card_viol = max(0.0, bsum - 3)
    card_resid = (bsum + sc - 3.0)

    return {
        "Jx": Jx,
        "Js": Js,
        "x1": x1, "x2": x2, "x3": x3, "x4": x4,
        "b6": b6v, "b7": b7v, "b8": b8v, "b9": b9v,
        "s6": s6, "s7": s7, "s8": s8, "s9": s9, "sc": sc,
        "obj": obj,
        "pen_energy": penE,
        "e1_resid": e1_resid,
        "e2_resid": e2_resid,
        "link_viol": link_viol,
        "link_resid_maxabs": link_resid_maxabs,
        "card_viol": card_viol,
        "card_resid": card_resid,
        "time_s": (t1 - t0),
        "stats": qubo_stats(bqm),
    }


def rolling_precision_large_instance_backtracking(
    J0_x: int = 1,
    J0_s: int = 1,
    Jstep: int = 1,
    Jmax_x: int = 4,
    Jmax_s: int = 4,
    lam_e1: float = 500.0,
    lam_e2: float = 500.0,
    lam_link: float = 500.0,
    lam_card: float = 500.0,
    sa: SAOptions = SAOptions(num_reads=300, sweeps=3000, seed=7),
    theta: float = 1e-10,
    max_iters: int = 100,
) -> Dict[str, Any]:
    """
    Rolling precision with backtracking over state (Jx, Js).
    Candidate ordering tries joint refinement first (often helps constraints):
        refine_xs, refine_x, refine_s, backtrack_xs, backtrack_x, backtrack_s
    """
    def valid(Jx: int, Js: int) -> bool:
        return (J0_x <= Jx <= Jmax_x) and (J0_s <= Js <= Jmax_s)

    def key(Jx: int, Js: int) -> Tuple[int, int]:
        return (Jx, Js)

    Jx, Js = J0_x, J0_s
    visited: Set[Tuple[int, int]] = {key(Jx, Js)}

    best = solve_large_instance_at_precision(Jx, Js, lam_e1, lam_e2, lam_link, lam_card, sa)
    best_pen = best["pen_energy"]

    history: List[Dict[str, Any]] = []
    history.append({"iter": 0, "move": "init", "var": None, **best})

    it = 0
    improved = True

    while improved and it < max_iters:
        it += 1
        improved = False

        candidates: List[Tuple[str, str, int, int]] = []

        # refinements
        candidates.append(("refine", "xs", Jx + Jstep, Js + Jstep))
        candidates.append(("refine", "x",  Jx + Jstep, Js))
        candidates.append(("refine", "s",  Jx, Js + Jstep))

        # backtracks
        candidates.append(("backtrack", "xs", Jx - Jstep, Js - Jstep))
        candidates.append(("backtrack", "x",  Jx - Jstep, Js))
        candidates.append(("backtrack", "s",  Jx, Js - Jstep))

        for move_type, var, Jx_c, Js_c in candidates:
            if not valid(Jx_c, Js_c):
                continue
            k = key(Jx_c, Js_c)
            if k in visited:
                continue
            visited.add(k)

            cand = solve_large_instance_at_precision(Jx_c, Js_c, lam_e1, lam_e2, lam_link, lam_card, sa)
            cand_pen = cand["pen_energy"]

            if cand_pen <= best_pen - theta:
                # accept
                Jx, Js = Jx_c, Js_c
                best = cand
                best_pen = cand_pen
                improved = True

                history.append({"iter": it, "move": move_type, "var": var, **cand})
                break

    monolithic = solve_large_instance_at_precision(Jmax_x, Jmax_s, lam_e1, lam_e2, lam_link, lam_card, sa)

    return {
        "history": history,
        "best": best,
        "best_penalized": best_pen,
        "best_J": {"Jx": Jx, "Js": Js},
        "visited_count": len(visited),
        "monolithic": monolithic,
    }


if __name__ == "__main__":
    report = rolling_precision_large_instance_backtracking(
        J0_x=1, J0_s=1,
        Jstep=1,
        Jmax_x=8, Jmax_s=8,
        lam_e1=500.0, lam_e2=500.0, lam_link=500.0, lam_card=500.0,
        sa=SAOptions(num_reads=300, sweeps=3000, seed=7),
        theta=1e-10,
        max_iters=50,
    )

    print("\n--- Rolling precision WITH backtracking (Large instance) ---")
    for rec in report["history"]:
        st = rec["stats"]
        print(
            f"it={rec['iter']:02d} move={rec['move']:<9} var={str(rec.get('var')):<2} "
            f"Jx={rec['Jx']} Js={rec['Js']} "
            f"x=[{rec['x1']:.6f},{rec['x2']:.6f},{rec['x3']:.6f},{rec['x4']:.6f}] "
            f"b=[{rec['b6']},{rec['b7']},{rec['b8']},{rec['b9']}] "
            f"obj={rec['obj']:.6e} penE={rec['pen_energy']:.6e} "
            f"e1={rec['e1_resid']:+.1e} e2={rec['e2_resid']:+.1e} "
            f"linkV={rec['link_viol']:.1e} linkR={rec['link_resid_maxabs']:.1e} "
            f"cardV={rec['card_viol']:.1e} cardR={rec['card_resid']:+.1e} "
            f"nvars={st['n_vars']} nquad={st['n_quadratic']} time={rec['time_s']:.3f}s"
        )

    m = report["monolithic"]
    st = m["stats"]
    print("\n--- Monolithic (max J) ---")
    print(
        f"Jx={m['Jx']} Js={m['Js']} "
        f"x=[{m['x1']:.6f},{m['x2']:.6f},{m['x3']:.6f},{m['x4']:.6f}] "
        f"b=[{m['b6']},{m['b7']},{m['b8']},{m['b9']}] "
        f"obj={m['obj']:.6e} penE={m['pen_energy']:.6e} "
        f"e1={m['e1_resid']:+.1e} e2={m['e2_resid']:+.1e} "
        f"linkV={m['link_viol']:.1e} linkR={m['link_resid_maxabs']:.1e} "
        f"cardV={m['card_viol']:.1e} cardR={m['card_resid']:+.1e} "
        f"nvars={st['n_vars']} nquad={st['n_quadratic']} time={m['time_s']:.3f}s"
    )



--- Rolling precision WITH backtracking (Large instance) ---
it=00 move=init      var=None Jx=1 Js=1 x=[0.300000,0.000000,0.400000,0.400000] b=[1,0,1,1] obj=1.720000e+00 penE=6.720000e+00 e1=+1.0e-01 e2=+1.8e-15 linkV=0.0e+00 linkR=1.1e-16 cardV=0.0e+00 cardR=+0.0e+00 nvars=49 nquad=406 time=0.045s
it=01 move=refine    var=xs Jx=2 Js=2 x=[0.340000,0.330000,0.360000,0.000000] b=[1,1,1,0] obj=3.077800e+00 penE=3.577800e+00 e1=+3.0e-02 e2=+1.0e-02 linkV=0.0e+00 linkR=2.2e-16 cardV=0.0e+00 cardR=+0.0e+00 nvars=85 nquad=1248 time=0.113s
it=02 move=refine    var=xs Jx=3 Js=3 x=[0.297000,0.264000,0.435000,0.004000] b=[1,1,1,0] obj=3.105000e+00 penE=3.117500e+00 e1=+0.0e+00 e2=+1.8e-15 linkV=4.0e-03 linkR=4.0e-03 cardV=0.0e+00 cardR=+0.0e+00 nvars=121 nquad=2554 time=0.234s
it=03 move=refine    var=xs Jx=4 Js=4 x=[0.498000,0.087000,0.434000,0.003100] b=[1,1,1,0] obj=2.824198e+00 penE=3.080653e+00 e1=+2.2e-02 e2=-3.3e-03 linkV=3.1e-03 linkR=3.1e-03 cardV=0.0e+00 cardR=+0.0e+00 nvars=157 nquad=